# Trust 질문 텍스트 분석

## 분석 목적

질문 피드백 중 잠재 안전 사유와 질문 문구 특성의 관계를 확인한다.

## 분석 범위

- 대상 채널: `QUESTION_FEEDBACK_OR_REPORT`
- `potential_safety_related_rule_flag`는 사용자가 선택한 피드백 사유이며 실제 위해 확정값이 아니다.
- 질문 노출 분모가 없으므로 결과를 신고율로 해석하지 않는다.
- 수동 라벨은 질문 주제 분류이며 안전 여부 라벨이 아니다.
- 후보 관계, 노출 횟수, 반복 노출 맥락은 분석 범위에 포함되지 않는다.
- 셀은 위에서부터 순서대로 실행한다.


## 0. 분석 기준

- 결과 변수: `potential_safety_related_rule_flag = 1`
- 분석 단위: 동일 사용자의 동일 정규화 문구를 한 건으로 축약한 `actor × text`
- 안전·비안전 사유가 함께 기록된 actor-text는 주 분석에서 제외하고 별도로 기록한다.
- 유사도 0.90 이상 문구를 semantic family로 묶어 데이터 누수를 막는다.
- 데이터 분할: semantic family 기준 Train 60% / Validation 20% / Test 20%
- 비교 모델: 다수 기준, 문자 TF-IDF, Kiwi 단어 TF-IDF, 문자+단어, Ko-SRoBERTa
- 평가 지표: PR-AUC, ROC-AUC, 균형정확도, F1, 상위 10% 정밀도
- 모델과 임계값은 Validation에서 선택하고 Test는 최종 평가에만 사용한다.
- 통계 검정은 효과크기, 95% 신뢰구간, BH-FDR 보정값을 함께 제시한다.

In [ ]:
from pathlib import Path
import json
import math
import re
import time
import unicodedata
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy import sparse
from scipy.stats import beta, fisher_exact, norm, wilcoxon
from sklearn.calibration import calibration_curve
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 260916
RNG = np.random.default_rng(SEED)


def find_project_root() -> Path:
    """현재 실행 위치에서 team7 프로젝트 루트를 찾는다."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "analysis" / "07_trust").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. team7 폴더 또는 그 하위에서 노트북을 실행하세요."
    )


ROOT = find_project_root()
ANALYSIS_DIR = ROOT / "analysis" / "07_trust"
OUTPUT_DIR = ANALYSIS_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
LABEL_REVIEW_DIR = OUTPUT_DIR / "labeling_review"

for directory in (FIG_DIR, TABLE_DIR, LABEL_REVIEW_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SAFETY_PATH = (
    ROOT / "data" / "processed" / "lifecycle_safety_attendance"
    / "mart_safety_event_v2_clean.csv.gz"
)
QUESTION_LOOKUP_PATH = (
    ROOT / "data" / "marts" / "23_marts_csv" / "mart_question_exposure.csv.gz"
)
TEXT_FEATURE_PATH = (
    ROOT / "analysis" / "04_content" / "outputs" / "unique_question_text_features_bert.csv"
)
TEXT_THEME_PATH = (
    ROOT / "analysis" / "04_content" / "outputs" / "final_deep"
    / "01c_text_level_adjusted_scores.csv"
)
EMBEDDING_PATH = (
    ROOT / "analysis" / "04_content" / "outputs" / "cache"
    / "ko_sroberta_question_embeddings.npz"
)
MANUAL_LABEL_PATH = LABEL_REVIEW_DIR / "trust_question_review_all_labeled_clean.csv"

REQUIRED_PATHS = {
    "안전 이벤트 마트": SAFETY_PATH,
    "질문 문구 조회 마트": QUESTION_LOOKUP_PATH,
    "텍스트 특징": TEXT_FEATURE_PATH,
    "텍스트 테마": TEXT_THEME_PATH,
    "SBERT 임베딩": EMBEDDING_PATH,
    "최종 수동 라벨": MANUAL_LABEL_PATH,
}
missing_paths = {name: path for name, path in REQUIRED_PATHS.items() if not path.exists()}
if missing_paths:
    missing_text = "\n".join(f"- {name}: {path}" for name, path in missing_paths.items())
    raise FileNotFoundError(f"필수 입력 파일이 없습니다.\n{missing_text}")

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Malgun Gothic")


def normalize_text(value):
    text = unicodedata.normalize("NFKC", str(value)).lower().strip()
    text = re.sub(r"[\s\u200b]+", " ", text)
    text = re.sub(r"[^0-9a-z가-힣 ]+", "", text)
    return re.sub(r"\s+", " ", text).strip()


def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close()
    return path


def bh_adjust(pvalues):
    p = np.asarray(pvalues, dtype=float)
    adjusted = np.full(len(p), np.nan)
    valid = np.isfinite(p)
    if valid.any():
        adjusted[valid] = multipletests(p[valid], method="fdr_bh")[1]
    return adjusted


def proportion_ci(success, total, alpha=0.05):
    success = np.asarray(success, dtype=float)
    total = np.asarray(total, dtype=float)
    lower = beta.ppf(alpha / 2, success + 0.5, total - success + 0.5)
    upper = beta.ppf(1 - alpha / 2, success + 0.5, total - success + 0.5)
    return lower, upper


def eval_binary(y, probability, threshold=0.5, sample_weight=None):
    y = np.asarray(y, dtype=int)
    probability = np.asarray(probability, dtype=float)
    prediction = probability >= threshold
    order = np.argsort(-probability)
    top_n = max(1, int(np.ceil(len(y) * 0.10)))
    if np.allclose(probability, probability[0]):
        top_precision = float(np.average(y, weights=sample_weight))
    else:
        weights = None if sample_weight is None else np.asarray(sample_weight)[order[:top_n]]
        top_precision = float(np.average(y[order[:top_n]], weights=weights))
    return {
        "pr_auc": average_precision_score(y, probability, sample_weight=sample_weight),
        "roc_auc": roc_auc_score(y, probability, sample_weight=sample_weight),
        "balanced_accuracy": balanced_accuracy_score(y, prediction, sample_weight=sample_weight),
        "f1": f1_score(y, prediction, sample_weight=sample_weight, zero_division=0),
        "precision": precision_score(y, prediction, sample_weight=sample_weight, zero_division=0),
        "recall": recall_score(y, prediction, sample_weight=sample_weight, zero_division=0),
        "brier": brier_score_loss(y, probability, sample_weight=sample_weight),
        "precision_at_top_10pct": top_precision,
        "threshold": threshold,
    }


def best_bacc_threshold(y, probability):
    candidates = np.unique(
        np.r_[np.linspace(0.05, 0.95, 181), np.quantile(probability, np.linspace(0, 1, 101))]
    )
    scores = [balanced_accuracy_score(y, probability >= threshold) for threshold in candidates]
    return float(candidates[int(np.argmax(scores))])


print(f"프로젝트 루트: {ROOT}")
print("분석 환경 준비 완료")

## 1. 데이터 연결 확인

In [ ]:
safety = pd.read_csv(SAFETY_PATH)
question_lookup = (
    pd.read_csv(QUESTION_LOOKUP_PATH, usecols=["question_id", "question_text"])
    .dropna(subset=["question_id", "question_text"])
    .drop_duplicates("question_id")
)
question_lookup["question_id"] = pd.to_numeric(question_lookup["question_id"], errors="coerce").astype("Int64")
question_lookup["normalized_text"] = question_lookup["question_text"].map(normalize_text)

feedback = safety.loc[safety["record_type"].eq("QUESTION_FEEDBACK_OR_REPORT")].copy()
feedback["question_id"] = pd.to_numeric(feedback["question_id"], errors="coerce").astype("Int64")
feedback["potential_safety"] = feedback["potential_safety_related_rule_flag"].fillna(0).astype(int)
feedback["event_at"] = pd.to_datetime(feedback["event_at_raw"], errors="coerce")
feedback["event_month"] = feedback["event_at"].dt.to_period("M").astype(str)
feedback = feedback.merge(
    question_lookup[["question_id", "question_text", "normalized_text"]],
    on="question_id", how="left", validate="many_to_one"
)

scope = pd.DataFrame([
    ("안전 마트 전체 행", len(safety)),
    ("질문 피드백·신고 채널 행", len(feedback)),
    ("질문 문구 연결 행", feedback["normalized_text"].notna().sum()),
    ("문구 미연결 행", feedback["normalized_text"].isna().sum()),
    ("피드백 사용자", feedback["actor_user_id"].nunique()),
    ("연결 질문 ID", feedback.loc[feedback["normalized_text"].notna(), "question_id"].nunique()),
    ("정규화 고유 문구", feedback["normalized_text"].nunique()),
    ("잠재 안전 사유 행", feedback["potential_safety"].sum()),
    ("분석 시작일", feedback["event_at"].min()),
    ("분석 종료일", feedback["event_at"].max()),
], columns=["항목", "값"])
scope.to_csv(TABLE_DIR / "01_scope_and_linkage_qa.csv", index=False, encoding="utf-8-sig")
display(scope)

assert len(feedback) == 51_424
assert feedback["normalized_text"].notna().sum() == 51_415
assert int(feedback["potential_safety"].sum()) == 5_694

질문 피드백 51,424건 중 잠재 안전 사유는 5,694건이다. 질문 문구 연결률은 99.98%다. 선호 불일치, 반복, 품질, 긍정 피드백은 잠재 안전 사유와 분리해 분석한다.

## 2. 피드백 사유 구성

In [ ]:
domain_order = [
    "PERSONAL_PREFERENCE", "CONTENT_SAFETY", "CONTENT_REPETITION",
    "POSITIVE_FEEDBACK", "RELATIONSHIP_FREQUENCY", "CONTENT_QUALITY", "OTHER"
]
domain_ko = {
    "PERSONAL_PREFERENCE": "개인 선호 불일치",
    "CONTENT_SAFETY": "잠재 안전",
    "CONTENT_REPETITION": "콘텐츠 반복",
    "POSITIVE_FEEDBACK": "긍정 피드백",
    "RELATIONSHIP_FREQUENCY": "관계·반복 접촉",
    "CONTENT_QUALITY": "콘텐츠 품질",
    "OTHER": "기타",
}
domain_counts = (
    feedback["feedback_domain_rule_v1"].value_counts()
    .reindex(domain_order).fillna(0).astype(int)
    .rename_axis("domain").reset_index(name="events")
)
domain_counts["label"] = domain_counts["domain"].map(domain_ko)
domain_counts["share_pct"] = domain_counts["events"] / domain_counts["events"].sum() * 100
domain_counts.to_csv(TABLE_DIR / "02_feedback_domain_composition.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(10, 5.5))
plot = domain_counts.sort_values("events")
colors = ["#E45756" if d == "CONTENT_SAFETY" else "#6B7A90" for d in plot["domain"]]
ax.barh(plot["label"], plot["share_pct"], color=colors)
for y, value in enumerate(plot["share_pct"]):
    ax.text(value + .35, y, f"{value:.1f}%", va="center")
ax.set_xlabel("질문 피드백 중 구성비")
ax.set_ylabel("")
ax.set_title("질문 피드백 채널은 안전 신고만으로 구성되지 않는다")
savefig("02_feedback_domain_composition.png")
display(domain_counts[["label", "events", "share_pct"]])

개인 선호 불일치가 73.9%로 가장 크고 잠재 안전 사유는 11.1%다. 피드백 총량만으로는 안전 문제와 추천·개인화 문제를 구분할 수 없다.

In [ ]:
monthly = pd.crosstab(feedback["event_month"], feedback["feedback_domain_rule_v1"])
monthly = monthly.reindex(columns=domain_order, fill_value=0)
monthly_share = monthly.div(monthly.sum(axis=1), axis=0) * 100
monthly_out = monthly.reset_index().merge(
    monthly_share.reset_index(), on="event_month", suffixes=("_events", "_share_pct")
)
monthly_out.to_csv(TABLE_DIR / "03_monthly_feedback_composition.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(12, 5.8))
palette = ["#8395A7", "#E45756", "#F2C14E", "#4ECDC4", "#A29BFE", "#55A868", "#B8B8B8"]
monthly_share.rename(columns=domain_ko).plot(kind="bar", stacked=True, ax=ax, color=palette, width=.85)
ax.set_ylabel("월별 피드백 구성비(%)")
ax.set_xlabel("")
ax.set_title("월별 피드백 사유 구성 — 5월 표본 집중을 함께 확인")
ax.legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
savefig("03_monthly_feedback_composition.png")

may_share = (feedback["event_month"] == "2023-05").mean() * 100
print(f"전체 질문 피드백 중 2023년 5월 비중: {may_share:.1f}%")

피드백이 2023년 5월에 집중돼 있으므로 전체기간 결과와 5월 결과를 함께 확인한다.

## 3. 반복 기록 정리

In [ ]:
linked = feedback.loc[feedback["normalized_text"].notna()].copy()
pair_raw = (
    linked.groupby(["actor_user_id", "normalized_text"], as_index=False)
    .agg(
        label_min=("potential_safety", "min"),
        label_max=("potential_safety", "max"),
        raw_rows=("potential_safety", "size"),
        first_event_at=("event_at", "min"),
        question_text=("question_text", "first"),
        question_id=("question_id", "first"),
    )
)
pair_raw["mixed_reason_pair"] = pair_raw["label_min"].ne(pair_raw["label_max"])
pair = pair_raw.loc[~pair_raw["mixed_reason_pair"]].copy()
pair["potential_safety"] = pair["label_max"].astype(int)
pair["event_month"] = pair["first_event_at"].dt.to_period("M").astype(str)

repeat_qa = pd.DataFrame([
    ("원시 연결 행", len(linked)),
    ("고유 actor × text", len(pair_raw)),
    ("반복으로 축약된 행", len(linked) - len(pair_raw)),
    ("동일 actor-text에 안전·비안전 사유가 모두 존재", pair_raw["mixed_reason_pair"].sum()),
    ("모델 주 분석의 순수 actor-text", len(pair)),
    ("주 분석 잠재 안전 actor-text", pair["potential_safety"].sum()),
], columns=["항목", "값"])
repeat_qa.to_csv(TABLE_DIR / "04_actor_text_dedup_qa.csv", index=False, encoding="utf-8-sig")
display(repeat_qa)

actor_concentration = linked.groupby("actor_user_id").size().sort_values(ascending=False)
print("사용자 1명당 질문 피드백 수 분위수")
display(actor_concentration.quantile([0, .5, .9, .95, .99, 1]).rename("feedback_rows").to_frame())

동일 사용자가 같은 문구에 남긴 반복 기록을 한 건으로 축약한다. 안전·비안전 사유가 함께 있는 actor-text는 주 분석에서 제외하고 별도로 기록한다.

## 4. 텍스트 자산 연결

In [ ]:
text_features = pd.read_csv(TEXT_FEATURE_PATH)
text_features = text_features.rename(columns={"question_text_normalized": "normalized_text"})
theme_map = pd.read_csv(TEXT_THEME_PATH, usecols=["question_text_normalized", "refined_theme_id", "theme_label_ko"]).drop_duplicates("question_text_normalized")
theme_map = theme_map.rename(columns={"question_text_normalized": "normalized_text"})
text_features = text_features.merge(theme_map, on="normalized_text", how="left")

emb_npz = np.load(EMBEDDING_PATH, allow_pickle=True)
embedding_texts = pd.Index(emb_npz["texts"].astype(str), name="normalized_text")
embeddings = emb_npz["embeddings"].astype("float32")
embedding_index = pd.Series(np.arange(len(embedding_texts)), index=embedding_texts)

pair = pair.merge(
    text_features[["normalized_text", "token_text", "semantic_group_090", "umap_x", "umap_y", "piece_count", "theme_label_ko", "refined_theme_id"]],
    on="normalized_text", how="left", validate="many_to_one"
)
missing_group = pair["semantic_group_090"].isna()
pair.loc[missing_group, "semantic_group_090"] = np.arange(missing_group.sum()) + 10_000_000
pair["semantic_group_090"] = pair["semantic_group_090"].astype(int)
pair["token_text"] = pair["token_text"].fillna(pair["normalized_text"])

asset_qa = pd.DataFrame([
    ("주 분석 actor-text", len(pair)),
    ("Kiwi 토큰 연결", pair["token_text"].notna().sum()),
    ("SBERT 임베딩 연결", pair["normalized_text"].isin(embedding_index.index).sum()),
    ("semantic family 연결", pair["semantic_group_090"].notna().sum()),
    ("탐색용 의미 테마 연결", pair["refined_theme_id"].notna().sum()),
], columns=["항목", "값"])
asset_qa.to_csv(TABLE_DIR / "05_text_asset_linkage_qa.csv", index=False, encoding="utf-8-sig")
display(asset_qa)

## 5. Train / Validation / Test 분할

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
pair["fold"] = -1
for fold, (_, idx) in enumerate(
    sgkf.split(pair["normalized_text"], pair["potential_safety"], groups=pair["semantic_group_090"])
):
    pair.iloc[idx, pair.columns.get_loc("fold")] = fold
pair["split"] = np.select(
    [pair["fold"].eq(0), pair["fold"].eq(1)],
    ["test", "validation"], default="train"
)
split_qa = (
    pair.groupby("split")
    .agg(rows=("potential_safety", "size"), safety=("potential_safety", "sum"),
         texts=("normalized_text", "nunique"), semantic_families=("semantic_group_090", "nunique"))
    .assign(safety_share=lambda d: d["safety"] / d["rows"])
    .reset_index()
)
split_qa.to_csv(TABLE_DIR / "06_fixed_split_qa.csv", index=False, encoding="utf-8-sig")
display(split_qa)

# 의미 가족이 두 split에 동시에 들어가지 않았는지 검증
assert pair.groupby("semantic_group_090")["split"].nunique().max() == 1

유사 문구가 서로 다른 데이터셋에 포함되지 않도록 semantic family 단위로 분할한다. 어휘 후보는 Train, 모델 선택은 Validation, 최종 평가는 Test에서 수행한다.

## 6. 질문 문구별 잠재 안전 구성

In [ ]:
text_stats = (
    pair.groupby("normalized_text", as_index=False)
    .agg(
        question_text=("question_text", "first"),
        feedback_pairs=("potential_safety", "size"),
        safety_pairs=("potential_safety", "sum"),
        reporters=("actor_user_id", "nunique"),
        piece_count=("piece_count", "first"),
        theme_label_ko=("theme_label_ko", "first"),
        refined_theme_id=("refined_theme_id", "first"),
        semantic_group_090=("semantic_group_090", "first"),
        umap_x=("umap_x", "first"), umap_y=("umap_y", "first"),
    )
)
p0 = pair["potential_safety"].mean()
prior_strength = 20
alpha0, beta0 = p0 * prior_strength, (1 - p0) * prior_strength
text_stats["raw_safety_share"] = text_stats["safety_pairs"] / text_stats["feedback_pairs"]
text_stats["eb_safety_share"] = (text_stats["safety_pairs"] + alpha0) / (text_stats["feedback_pairs"] + prior_strength)
text_stats["eb_ci_low"] = beta.ppf(.025, text_stats["safety_pairs"] + alpha0, text_stats["feedback_pairs"] - text_stats["safety_pairs"] + beta0)
text_stats["eb_ci_high"] = beta.ppf(.975, text_stats["safety_pairs"] + alpha0, text_stats["feedback_pairs"] - text_stats["safety_pairs"] + beta0)
text_stats["high_confidence_above_global"] = text_stats["eb_ci_low"] > p0
text_stats.to_csv(TABLE_DIR / "07_question_text_safety_composition_eb.csv", index=False, encoding="utf-8-sig")

top_volume = text_stats.sort_values(["safety_pairs", "feedback_pairs"], ascending=False).head(15)
top_composition = text_stats.loc[text_stats["feedback_pairs"].ge(10)].sort_values("eb_ci_low", ascending=False).head(15)
top_volume.to_csv(TABLE_DIR / "07b_top_safety_volume_texts.csv", index=False, encoding="utf-8-sig")
top_composition.to_csv(TABLE_DIR / "07c_high_safety_composition_texts.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(10.5, 6.2))
sc = ax.scatter(
    text_stats["feedback_pairs"], text_stats["eb_safety_share"] * 100,
    c=np.log1p(text_stats["safety_pairs"]), cmap="magma", alpha=.55, s=22
)
ax.set_xscale("log")
ax.axhline(p0 * 100, color="#333333", linestyle="--", label=f"전체 기준 {p0*100:.1f}%")
ax.set_xlabel("고유 actor-text 피드백 수 (로그 축)")
ax.set_ylabel("잠재 안전 사유 EB 구성비(%)")
ax.set_title("신고 건수가 많은 질문과 안전 구성비가 높은 질문은 같지 않다")
ax.legend()
plt.colorbar(sc, ax=ax, label="잠재 안전 actor-text 수(log1p)")
savefig("07_question_volume_vs_safety_composition.png")

print("잠재 안전 건수 상위 문구")
display(top_volume[["question_text", "feedback_pairs", "safety_pairs", "eb_safety_share"]].head(10))
print("피드백 10건 이상 중 안전 구성비 하한이 높은 문구")
display(top_composition[["question_text", "feedback_pairs", "safety_pairs", "eb_safety_share", "eb_ci_low"]].head(10))

건수와 잠재 안전 구성비를 분리해 본다. 소표본의 극단값을 줄이기 위해 EB 추정치와 신뢰구간을 사용한다.

In [ ]:
safety_sorted = text_stats.sort_values("safety_pairs", ascending=False).reset_index(drop=True)
safety_sorted["text_rank"] = np.arange(1, len(safety_sorted) + 1)
safety_sorted["text_share"] = safety_sorted["text_rank"] / len(safety_sorted)
safety_sorted["cumulative_safety_share"] = safety_sorted["safety_pairs"].cumsum() / safety_sorted["safety_pairs"].sum()
safety_sorted.to_csv(TABLE_DIR / "08_safety_signal_pareto_curve.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.plot(safety_sorted["text_share"] * 100, safety_sorted["cumulative_safety_share"] * 100, color="#E45756", linewidth=2.5)
ax.plot([0, 100], [0, 100], color="#999999", linestyle="--", linewidth=1)
ax.set_xlabel("잠재 안전 건수 순으로 누적한 문구 비중(%)")
ax.set_ylabel("누적 잠재 안전 actor-text 비중(%)")
ax.set_title("잠재 안전 신호는 일부 질문 문구에 집중된다")
savefig("08_safety_signal_pareto.png")

for pct in [1, 5, 10]:
    k = max(1, int(len(safety_sorted) * pct / 100))
    captured = safety_sorted.head(k)["safety_pairs"].sum() / safety_sorted["safety_pairs"].sum() * 100
    print(f"상위 {pct}% 문구가 잠재 안전 actor-text의 {captured:.1f}%를 포함")

잠재 안전 피드백은 일부 질문 문구에 집중된다. 노출 분모가 없으므로 이를 질문별 위험률로 해석하지 않는다.

## 7. 위험 표현 사전

In [ ]:
# 결과 라벨과 무관하게 Trust 검토에서 필요한 표현 축을 먼저 고정한다.
LEXICON = {
    "사생활·감시 추정": ["염탐", "몰폰", "스토킹", "몰래 보", "몰래볼", "sns 훔쳐", "비밀을 볼"],
    "체취·위생·신체 희화": ["발냄새", "땀냄새", "냄새", "방귀", "발가락", "콧구멍", "겨드랑이", "코딱지", "콧물", "백태", "트름", "더럽", "씻지", "목욕 안"],
    "연애·친밀행동 추정": ["키스", "스킨십", "성관계", "야한", "19금", "모텔", "첫경험", "남친", "여친", "전남친", "전여친", "플러팅", "양다리", "사귀", "호감", "신체접촉"],
    "외모·신체 평가": ["몸매", "외모", "얼굴", "어깨", "등빨", "가슴", "뚱뚱", "살찐", "못생", "마스크", "섹시", "근육", "피부", "매력적인", "화장"],
    "부정 낙인·조롱": ["찐따", "아싸", "바보", "멍청", "관종", "허당", "재수없", "눈치없", "민폐", "최악", "왕따", "일진", "폭력", "범죄", "중독"],
    "맥락 불명·강제 상상": ["이모티콘만 보고", "알아서 상상", "상상해서 골라", "vote"],
    "음식·섭취 취향 추정": ["먹방", "토스트", "와플", "탕후루", "음식을", "먹을 것", "먹는 사람"],
    "배려·신뢰·친사회": ["배려", "믿음", "믿을", "챙김", "챙겨", "고민을", "위로", "의리", "친절"],
}
CORE_RISK_CATEGORIES = ["사생활·감시 추정", "체취·위생·신체 희화", "연애·친밀행동 추정", "외모·신체 평가"]
def lexicon_flag(text, words):
    return int(any(word in str(text) for word in words))

for category, words in LEXICON.items():
    pair[f"lex_{category}"] = pair["normalized_text"].map(lambda x: lexicon_flag(x, words))
    text_stats[f"lex_{category}"] = text_stats["normalized_text"].map(lambda x: lexicon_flag(x, words))
lex_cols = [f"lex_{c}" for c in LEXICON]
core_cols = [f"lex_{c}" for c in CORE_RISK_CATEGORIES]
pair["lex_any"] = pair[core_cols].max(axis=1)
text_stats["lex_any"] = text_stats[core_cols].max(axis=1)

lex_rows = []
for category in LEXICON:
    col = f"lex_{category}"
    for split_name in ["all", "test"]:
        sub = pair if split_name == "all" else pair.loc[pair["split"].eq("test")]
        present = sub[col].eq(1)
        a = int(sub.loc[present, "potential_safety"].sum())
        b = int(present.sum() - a)
        c = int(sub.loc[~present, "potential_safety"].sum())
        d = int((~present).sum() - c)
        odds_ratio, pvalue = fisher_exact([[a, b], [c, d]]) if (a + b) > 0 else (np.nan, np.nan)
        rate_present = a / (a + b) if a + b else np.nan
        rate_absent = c / (c + d) if c + d else np.nan
        se = math.sqrt(sum(1 / (v + .5) for v in [a, b, c, d])) if (a + b) > 0 else np.nan
        log_or = math.log((a + .5) * (d + .5) / ((b + .5) * (c + .5))) if (a + b) > 0 else np.nan
        lex_rows.append({
            "split": split_name, "category": category, "rows_with_flag": a + b,
            "safety_share_with": rate_present, "safety_share_without": rate_absent,
            "delta_pp": (rate_present - rate_absent) * 100,
            "odds_ratio": math.exp(log_or),
            "or_ci_low": math.exp(log_or - 1.96 * se),
            "or_ci_high": math.exp(log_or + 1.96 * se),
            "p_value": pvalue,
        })
lexicon_results = pd.DataFrame(lex_rows)
lexicon_results["q_value"] = lexicon_results.groupby("split")["p_value"].transform(bh_adjust)
lexicon_results.to_csv(TABLE_DIR / "09_seed_lexicon_effects.csv", index=False, encoding="utf-8-sig")

test_lex = lexicon_results.loc[lexicon_results["split"].eq("test")].sort_values("odds_ratio")
test_lex_plot = test_lex.dropna(subset=["odds_ratio", "or_ci_low", "or_ci_high"])
fig, ax = plt.subplots(figsize=(9.5, 5.7))
y = np.arange(len(test_lex_plot))
ax.errorbar(
    test_lex_plot["odds_ratio"], y,
    xerr=[test_lex_plot["odds_ratio"] - test_lex_plot["or_ci_low"], test_lex_plot["or_ci_high"] - test_lex_plot["odds_ratio"]],
    fmt="o", color="#E45756", ecolor="#888888", capsize=3
)
ax.axvline(1, color="#333333", linestyle="--")
ax.set_yticks(y, test_lex_plot["category"])
ax.set_xscale("log")
ax.set_xlabel("잠재 안전 사유 Odds Ratio (Test, 로그 축)")
ax.set_title("사전 고정 표현 범주의 잠재 안전 사유 구성 차이")
savefig("09_seed_lexicon_test_effects.png")
display(test_lex[["category", "rows_with_flag", "safety_share_with", "safety_share_without", "delta_pp", "odds_ratio", "or_ci_low", "or_ci_high", "q_value"]])

위험 표현 사전은 문구 유형과 잠재 안전 사유의 관계를 확인하기 위한 기준이다. 효과는 Test 데이터에서 확인하고 다중검정 결과는 FDR로 보정한다.

## 8. 표현 범주 효과

In [ ]:
glm_data = pair.copy()
glm_data["text_length_z"] = (
    glm_data["normalized_text"].str.len() - glm_data["normalized_text"].str.len().mean()
) / glm_data["normalized_text"].str.len().std()
month_dummies = pd.get_dummies(glm_data["event_month"], prefix="month", drop_first=True, dtype=float)
X_glm = pd.concat([glm_data[lex_cols].astype(float), glm_data[["text_length_z"]], month_dummies], axis=1)
X_glm = sm.add_constant(X_glm, has_constant="add")
glm_fit = sm.GLM(
    glm_data["potential_safety"].astype(float), X_glm,
    family=sm.families.Binomial()
).fit(cov_type="cluster", cov_kwds={"groups": glm_data["semantic_group_090"]})

glm_rows = []
for category in LEXICON:
    term = f"lex_{category}"
    coef, se, pvalue = glm_fit.params[term], glm_fit.bse[term], glm_fit.pvalues[term]
    glm_rows.append({
        "category": category,
        "adjusted_odds_ratio": math.exp(coef),
        "or_ci_low": math.exp(coef - 1.96 * se),
        "or_ci_high": math.exp(coef + 1.96 * se),
        "p_value": pvalue,
    })
glm_effects = pd.DataFrame(glm_rows)
glm_effects["q_value"] = bh_adjust(glm_effects["p_value"])
glm_effects.to_csv(TABLE_DIR / "10_adjusted_lexicon_glm.csv", index=False, encoding="utf-8-sig")

plot = glm_effects.sort_values("adjusted_odds_ratio")
fig, ax = plt.subplots(figsize=(9.5, 6))
y = np.arange(len(plot))
ax.errorbar(
    plot["adjusted_odds_ratio"], y,
    xerr=[plot["adjusted_odds_ratio"] - plot["or_ci_low"], plot["or_ci_high"] - plot["adjusted_odds_ratio"]],
    fmt="o", color="#C44E52", ecolor="#888888", capsize=3
)
ax.axvline(1, color="#333333", linestyle="--")
ax.set_yticks(y, plot["category"])
ax.set_xscale("log")
ax.set_xlabel("조정 Odds Ratio (로그 축)")
ax.set_title("월·문장 길이를 통제해도 표현 범주 차이가 남는가")
savefig("10_adjusted_lexicon_glm.png")
display(glm_effects.sort_values("adjusted_odds_ratio", ascending=False))

월과 문장 길이를 통제하고 semantic family 기준 군집 표준오차를 적용한다. 결과는 피드백 내부에서 잠재 안전 사유가 선택될 odds의 차이로 해석한다.

## 9. Kiwi 형태소·2-gram 재검증

In [ ]:
def term_enrichment(frame, vectorizer=None, fit=False):
    if fit:
        vectorizer = CountVectorizer(
            tokenizer=str.split, token_pattern=None, lowercase=False,
            ngram_range=(1, 2), binary=True, min_df=20, max_features=20_000
        )
        X = vectorizer.fit_transform(frame["token_text"].fillna(""))
    else:
        X = vectorizer.transform(frame["token_text"].fillna(""))
    terms = np.asarray(vectorizer.get_feature_names_out())
    y = frame["potential_safety"].to_numpy(dtype=int)
    pos = y == 1
    neg = ~pos
    a = np.asarray(X[pos].sum(axis=0)).ravel().astype(float)
    c = np.asarray(X[neg].sum(axis=0)).ravel().astype(float)
    b = pos.sum() - a
    d = neg.sum() - c
    log_or = np.log((a + .5) * (d + .5) / ((b + .5) * (c + .5)))
    se = np.sqrt(1/(a+.5) + 1/(b+.5) + 1/(c+.5) + 1/(d+.5))
    z = log_or / se
    p = 2 * norm.sf(np.abs(z))
    rate_with = a / np.maximum(a + c, 1)
    rate_without = b / np.maximum(b + d, 1)
    stats = pd.DataFrame({
        "term": terms, "safety_with": a, "other_with": c,
        "rows_with": a + c, "safety_share_with": rate_with,
        "safety_share_without": rate_without,
        "delta_pp": (rate_with - rate_without) * 100,
        "odds_ratio": np.exp(log_or),
        "or_ci_low": np.exp(log_or - 1.96*se),
        "or_ci_high": np.exp(log_or + 1.96*se),
        "p_value": p,
    })
    stats["q_value"] = bh_adjust(stats["p_value"])

    unique_text = frame.drop_duplicates("normalized_text")
    Xu = vectorizer.transform(unique_text["token_text"].fillna(""))
    stats["unique_texts_with"] = np.asarray(Xu.sum(axis=0)).ravel()
    return stats, vectorizer

train = pair.loc[pair["split"].eq("train")].copy()
validation = pair.loc[pair["split"].eq("validation")].copy()
test = pair.loc[pair["split"].eq("test")].copy()

train_terms, term_vectorizer = term_enrichment(train, fit=True)
candidate_terms = (
    train_terms.loc[
        train_terms["unique_texts_with"].ge(3)
        & train_terms["q_value"].lt(.05)
        & train_terms["odds_ratio"].ge(1.30)
        & train_terms["delta_pp"].ge(1.0)
    ]
    .sort_values(["delta_pp", "odds_ratio"], ascending=False)
    .head(30)["term"].tolist()
)
test_terms, _ = term_enrichment(test, vectorizer=term_vectorizer, fit=False)
heldout_terms = test_terms.loc[test_terms["term"].isin(candidate_terms)].copy()
heldout_terms["q_value_selected"] = bh_adjust(heldout_terms["p_value"])
heldout_terms["replicated"] = (
    heldout_terms["q_value_selected"].lt(.05)
    & heldout_terms["odds_ratio"].ge(1.30)
    & heldout_terms["delta_pp"].gt(0)
    & heldout_terms["unique_texts_with"].ge(3)
)
heldout_terms = heldout_terms.sort_values(["replicated", "odds_ratio"], ascending=False)
train_terms.to_csv(TABLE_DIR / "11_train_token_enrichment_all.csv", index=False, encoding="utf-8-sig")
heldout_terms.to_csv(TABLE_DIR / "11b_heldout_token_replication.csv", index=False, encoding="utf-8-sig")

plot_terms = heldout_terms.loc[heldout_terms["rows_with"].gt(0)].head(18).sort_values("odds_ratio")
fig, ax = plt.subplots(figsize=(9.5, max(5.5, .34 * len(plot_terms))))
colors = np.where(plot_terms["replicated"], "#E45756", "#A0A0A0")
ax.barh(plot_terms["term"], plot_terms["odds_ratio"], color=colors)
ax.axvline(1, color="#333333", linestyle="--")
ax.set_xlabel("Test Odds Ratio")
ax.set_title("Train에서 발견한 표현이 새로운 의미 가족에서도 재현되는가")
savefig("11_heldout_token_replication.png")
print(f"Train 후보 {len(candidate_terms)}개 중 Test 재현 기준 충족: {int(heldout_terms['replicated'].sum())}개")
display(heldout_terms[["term", "unique_texts_with", "rows_with", "safety_share_with", "delta_pp", "odds_ratio", "or_ci_low", "or_ci_high", "q_value_selected", "replicated"]].head(20))

Train에서 형태소와 2-gram 후보를 찾고 Test에서 같은 방향이 재현되는지 확인한다. 특정 질문에만 나타난 표현은 제외하며, 재현된 표현은 검수 우선순위 신호로 사용한다.

## 10. 임베딩 의미 분포

In [ ]:
semantic_plot = text_stats.dropna(subset=["umap_x", "umap_y"]).copy()
fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(
    semantic_plot["umap_x"], semantic_plot["umap_y"],
    c=semantic_plot["eb_safety_share"] * 100,
    s=12 + np.log1p(semantic_plot["feedback_pairs"]) * 12,
    cmap="RdYlBu_r", vmin=0, vmax=min(50, semantic_plot["eb_safety_share"].quantile(.98) * 100),
    alpha=.70, linewidth=0
)
top_annotate = semantic_plot.nlargest(10, "safety_pairs")
for _, row in top_annotate.iterrows():
    ax.annotate(str(row["question_text"])[:16], (row["umap_x"], row["umap_y"]), fontsize=7, alpha=.8)
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel("SBERT 의미 공간(UMAP 시각화)")
ax.set_title("잠재 안전 신호는 하나의 독립된 주제가 아니라 여러 의미 영역에 분산된다")
plt.colorbar(sc, ax=ax, label="잠재 안전 사유 EB 구성비(%)")
savefig("12_semantic_map_safety_composition.png")

theme_safety = (
    pair.dropna(subset=["theme_label_ko"])
    .groupby(["refined_theme_id", "theme_label_ko"], as_index=False)
    .agg(feedback_pairs=("potential_safety", "size"), safety_pairs=("potential_safety", "sum"), texts=("normalized_text", "nunique"))
)
theme_safety["safety_share"] = theme_safety["safety_pairs"] / theme_safety["feedback_pairs"]
theme_safety["eb_safety_share"] = (theme_safety["safety_pairs"] + alpha0) / (theme_safety["feedback_pairs"] + prior_strength)
theme_safety.to_csv(TABLE_DIR / "12b_exploratory_semantic_theme_safety.csv", index=False, encoding="utf-8-sig")
display(theme_safety.sort_values("eb_safety_share", ascending=False).head(10))

잠재 안전 피드백은 체취, 외모, 친밀관계, 감시 등 여러 의미 영역에 분산돼 있다. 의미 테마는 탐색용으로 사용하고 검증 데이터와 개별 문구를 함께 확인한다.

## 11. 유사 질문 비교

In [ ]:
# 핵심 위험 표현이 있는 문구와 없는 문구를 같은 의미 테마 안에서 SBERT 최근접 매칭한다.
match_pool = text_stats.loc[text_stats["feedback_pairs"].ge(5)].copy()
match_pool["emb_idx"] = match_pool["normalized_text"].map(embedding_index)
match_pool = match_pool.dropna(subset=["emb_idx", "refined_theme_id"]).copy()
match_pool["emb_idx"] = match_pool["emb_idx"].astype(int)
risk_pool = match_pool.loc[match_pool["lex_any"].eq(1)].copy()
control_pool = match_pool.loc[match_pool["lex_any"].eq(0)].copy()

pairs = []
for theme_id, risk_group in risk_pool.groupby("refined_theme_id"):
    controls = control_pool.loc[control_pool["refined_theme_id"].eq(theme_id)].copy()
    if controls.empty:
        continue
    C = embeddings[controls["emb_idx"].to_numpy()]
    C = C / np.maximum(np.linalg.norm(C, axis=1, keepdims=True), 1e-12)
    R = embeddings[risk_group["emb_idx"].to_numpy()]
    R = R / np.maximum(np.linalg.norm(R, axis=1, keepdims=True), 1e-12)
    sims = R @ C.T
    for i, (_, rr) in enumerate(risk_group.iterrows()):
        order = np.argsort(-sims[i])
        chosen = None
        for j in order[:20]:
            cc = controls.iloc[j]
            if abs(len(rr["normalized_text"]) - len(cc["normalized_text"])) > 18:
                continue
            if abs(np.log1p(rr["feedback_pairs"]) - np.log1p(cc["feedback_pairs"])) > 1.5:
                continue
            if sims[i, j] < .60:
                continue
            chosen = cc
            break
        if chosen is not None:
            pairs.append({
                "risk_text": rr["question_text"], "control_text": chosen["question_text"],
                "cosine": float(sims[i, controls.index.get_loc(chosen.name)]),
                "risk_feedback": rr["feedback_pairs"], "control_feedback": chosen["feedback_pairs"],
                "risk_eb_share": rr["eb_safety_share"], "control_eb_share": chosen["eb_safety_share"],
                "delta_pp": (rr["eb_safety_share"] - chosen["eb_safety_share"]) * 100,
                "theme_label_ko": rr["theme_label_ko"],
            })
matched_pairs = pd.DataFrame(pairs)
if not matched_pairs.empty:
    stat, pvalue = wilcoxon(matched_pairs["risk_eb_share"], matched_pairs["control_eb_share"], alternative="greater")
    boot = []
    for _ in range(2000):
        sample = matched_pairs.sample(len(matched_pairs), replace=True, random_state=int(RNG.integers(0, 2**31-1)))
        boot.append(sample["delta_pp"].mean())
    match_summary = pd.DataFrame([{
        "matched_pairs": len(matched_pairs),
        "mean_delta_pp": matched_pairs["delta_pp"].mean(),
        "median_delta_pp": matched_pairs["delta_pp"].median(),
        "bootstrap_ci_low": np.quantile(boot, .025),
        "bootstrap_ci_high": np.quantile(boot, .975),
        "wilcoxon_one_sided_p": pvalue,
        "median_cosine": matched_pairs["cosine"].median(),
    }])
else:
    match_summary = pd.DataFrame([{"matched_pairs": 0}])
matched_pairs.to_csv(TABLE_DIR / "13_semantic_matched_pairs.csv", index=False, encoding="utf-8-sig")
match_summary.to_csv(TABLE_DIR / "13b_semantic_matched_pair_summary.csv", index=False, encoding="utf-8-sig")
display(match_summary)
if not matched_pairs.empty:
    display(matched_pairs.sort_values("delta_pp", ascending=False).head(12))

주제, 문장 길이, 피드백 규모가 비슷한 질문을 매칭해 표현 차이를 비교한다. 관찰 데이터 기반의 탐색적 결과이므로 인과효과로 해석하지 않는다.

## 12. 텍스트 분류 성능

In [ ]:
train_idx = pair.index[pair["split"].eq("train")]
val_idx = pair.index[pair["split"].eq("validation")]
test_idx = pair.index[pair["split"].eq("test")]

y_train = pair.loc[train_idx, "potential_safety"].to_numpy()
y_val = pair.loc[val_idx, "potential_safety"].to_numpy()
y_test = pair.loc[test_idx, "potential_safety"].to_numpy()

model_objects = {}
validation_predictions = {}
registry = []

def register_validation(name, y, p, threshold, seconds):
    row = {"model": name, "fit_seconds": seconds, **eval_binary(y, p, threshold)}
    registry.append(row)

# 기준선
base_prob = float(y_train.mean())
p_base_val = np.repeat(base_prob, len(y_val))
register_validation("다수비율 기준선", y_val, p_base_val, .5, 0.0)
validation_predictions["다수비율 기준선"] = p_base_val

# 문자 TF-IDF
t0 = time.time()
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=50_000, sublinear_tf=True)
Xc_train = char_vec.fit_transform(pair.loc[train_idx, "normalized_text"])
Xc_val = char_vec.transform(pair.loc[val_idx, "normalized_text"])
char_model = LogisticRegression(max_iter=1500, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED)
char_model.fit(Xc_train, y_train)
p_char_val = char_model.predict_proba(Xc_val)[:, 1]
t_char = best_bacc_threshold(y_val, p_char_val)
register_validation("문자 TF-IDF", y_val, p_char_val, t_char, time.time()-t0)
validation_predictions["문자 TF-IDF"] = p_char_val
model_objects["문자 TF-IDF"] = (char_vec, char_model)

# Kiwi 형태소 TF-IDF
t0 = time.time()
word_vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2, max_features=30_000, sublinear_tf=True)
Xw_train = word_vec.fit_transform(pair.loc[train_idx, "token_text"].fillna(""))
Xw_val = word_vec.transform(pair.loc[val_idx, "token_text"].fillna(""))
word_model = LogisticRegression(max_iter=1500, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED)
word_model.fit(Xw_train, y_train)
p_word_val = word_model.predict_proba(Xw_val)[:, 1]
t_word = best_bacc_threshold(y_val, p_word_val)
register_validation("Kiwi 단어 TF-IDF", y_val, p_word_val, t_word, time.time()-t0)
validation_predictions["Kiwi 단어 TF-IDF"] = p_word_val
model_objects["Kiwi 단어 TF-IDF"] = (word_vec, word_model)

# 문자 + Kiwi 결합
t0 = time.time()
Xcw_train = sparse.hstack([Xc_train, Xw_train], format="csr")
Xcw_val = sparse.hstack([Xc_val, Xw_val], format="csr")
combo_model = LogisticRegression(max_iter=1500, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED)
combo_model.fit(Xcw_train, y_train)
p_combo_val = combo_model.predict_proba(Xcw_val)[:, 1]
t_combo = best_bacc_threshold(y_val, p_combo_val)
register_validation("문자+Kiwi TF-IDF", y_val, p_combo_val, t_combo, time.time()-t0)
validation_predictions["문자+Kiwi TF-IDF"] = p_combo_val
model_objects["문자+Kiwi TF-IDF"] = ((char_vec, word_vec), combo_model)

# Ko-SRoBERTa 임베딩
def embedding_matrix(frame):
    idx = frame["normalized_text"].map(embedding_index)
    out = np.zeros((len(frame), embeddings.shape[1]), dtype="float32")
    present = idx.notna().to_numpy()
    if present.any():
        out[present] = embeddings[idx.loc[idx.notna()].astype(int).to_numpy()]
    return out

t0 = time.time()
Xe_train = embedding_matrix(pair.loc[train_idx])
Xe_val = embedding_matrix(pair.loc[val_idx])
emb_model = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED)
emb_model.fit(Xe_train, y_train)
p_emb_val = emb_model.predict_proba(Xe_val)[:, 1]
t_emb = best_bacc_threshold(y_val, p_emb_val)
register_validation("Ko-SRoBERTa", y_val, p_emb_val, t_emb, time.time()-t0)
validation_predictions["Ko-SRoBERTa"] = p_emb_val
model_objects["Ko-SRoBERTa"] = (None, emb_model)

model_registry = pd.DataFrame(registry).sort_values("pr_auc", ascending=False)
model_registry.to_csv(TABLE_DIR / "14_model_validation_registry.csv", index=False, encoding="utf-8-sig")
display(model_registry)

fig, ax = plt.subplots(figsize=(10, 5.8))
plot = model_registry.sort_values("pr_auc")
ax.barh(plot["model"], plot["pr_auc"], color=np.where(plot["model"].eq("다수비율 기준선"), "#999999", "#4C72B0"))
ax.axvline(y_val.mean(), color="#E45756", linestyle="--", label=f"Validation 유병률 {y_val.mean():.3f}")
ax.set_xlabel("Validation PR-AUC")
ax.set_title("텍스트 표현 방식별 잠재 안전 사유 선별 성능")
ax.legend()
savefig("14_model_validation_pr_auc.png")

모델은 질문의 실제 위험 여부가 아니라 피드백 사유가 잠재 안전인지 구분한다. Validation PR-AUC로 모델을 선택하고 Test에서 최종 평가한다.

In [ ]:
selected_name = model_registry.loc[model_registry["model"].ne("다수비율 기준선")].iloc[0]["model"]
selected_threshold = float(model_registry.loc[model_registry["model"].eq(selected_name), "threshold"].iloc[0])
trainval = pair.loc[pair["split"].isin(["train", "validation"])].copy()
y_trainval = trainval["potential_safety"].to_numpy()
test_frame = pair.loc[test_idx].copy()

t0 = time.time()
if selected_name == "문자 TF-IDF":
    final_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=3, max_features=50_000, sublinear_tf=True)
    Xtr = final_char.fit_transform(trainval["normalized_text"])
    Xte = final_char.transform(test_frame["normalized_text"])
    final_model = LogisticRegression(max_iter=1500, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED).fit(Xtr, y_trainval)
elif selected_name == "Kiwi 단어 TF-IDF":
    final_word = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, ngram_range=(1,2), min_df=2, max_features=30_000, sublinear_tf=True)
    Xtr = final_word.fit_transform(trainval["token_text"].fillna(""))
    Xte = final_word.transform(test_frame["token_text"].fillna(""))
    final_model = LogisticRegression(max_iter=1500, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED).fit(Xtr, y_trainval)
elif selected_name == "문자+Kiwi TF-IDF":
    final_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=3, max_features=50_000, sublinear_tf=True)
    final_word = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, ngram_range=(1,2), min_df=2, max_features=30_000, sublinear_tf=True)
    Xtr = sparse.hstack([final_char.fit_transform(trainval["normalized_text"]), final_word.fit_transform(trainval["token_text"].fillna(""))], format="csr")
    Xte = sparse.hstack([final_char.transform(test_frame["normalized_text"]), final_word.transform(test_frame["token_text"].fillna(""))], format="csr")
    final_model = LogisticRegression(max_iter=1500, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED).fit(Xtr, y_trainval)
else:
    Xtr = embedding_matrix(trainval)
    Xte = embedding_matrix(test_frame)
    final_model = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, solver="liblinear", random_state=SEED).fit(Xtr, y_trainval)

p_test = final_model.predict_proba(Xte)[:, 1]
final_metrics = eval_binary(y_test, p_test, selected_threshold)
baseline_metrics = eval_binary(y_test, np.repeat(y_trainval.mean(), len(y_test)), .5)

# semantic family 단위 bootstrap CI
test_groups = test_frame["semantic_group_090"].to_numpy()
unique_groups = np.unique(test_groups)
boot_metrics = []
for _ in range(300):
    sampled = RNG.choice(unique_groups, size=len(unique_groups), replace=True)
    idx_parts = [np.flatnonzero(test_groups == g) for g in sampled]
    idx_b = np.concatenate(idx_parts)
    if len(np.unique(y_test[idx_b])) < 2:
        continue
    boot_metrics.append(eval_binary(y_test[idx_b], p_test[idx_b], selected_threshold))
boot_df = pd.DataFrame(boot_metrics)
metric_rows = []
for metric, value in final_metrics.items():
    if metric == "threshold":
        continue
    metric_rows.append({
        "model": selected_name, "metric": metric, "estimate": value,
        "ci_low": boot_df[metric].quantile(.025), "ci_high": boot_df[metric].quantile(.975),
        "baseline": baseline_metrics.get(metric, np.nan)
    })
final_test = pd.DataFrame(metric_rows)
final_test["selected_threshold"] = selected_threshold
final_test["fit_seconds"] = time.time() - t0
final_test.to_csv(TABLE_DIR / "15_final_untouched_test_metrics.csv", index=False, encoding="utf-8-sig")
display(final_test)

precision, recall, _ = precision_recall_curve(y_test, p_test)
fig, ax = plt.subplots(figsize=(7.5, 5.8))
ax.plot(recall, precision, color="#4C72B0", linewidth=2.3, label=f"{selected_name} (AP={final_metrics['pr_auc']:.3f})")
ax.axhline(y_test.mean(), color="#E45756", linestyle="--", label=f"기준선={y_test.mean():.3f}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("완전 미사용 Test 의미 가족 — Precision-Recall")
ax.legend()
savefig("15_final_test_precision_recall.png")

prob_true, prob_pred = calibration_curve(y_test, p_test, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(6.8, 5.8))
ax.plot(prob_pred, prob_true, marker="o", color="#4C72B0")
ax.plot([0,1], [0,1], linestyle="--", color="#777777")
ax.set_xlabel("예측 확률")
ax.set_ylabel("실제 잠재 안전 구성")
ax.set_title("Test calibration — 자동 판정 가능성 점검")
savefig("15b_final_test_calibration.png")
print(f"선택 모델: {selected_name}, Validation에서 고정한 임계값: {selected_threshold:.3f}")

PR-AUC는 잠재 안전 비중을 기준선으로 비교한다. 상위 10% 정밀도와 재현율을 함께 확인해 검수 우선순위 신호로 사용할 수 있는지 판단한다.

## 13. 동일 문구의 사유 혼합

In [ ]:
text_mix = (
    linked.groupby("normalized_text", as_index=False)
    .agg(
        events=("potential_safety", "size"),
        safety_events=("potential_safety", "sum"),
        question_text=("question_text", "first")
    )
)
text_mix["other_events"] = text_mix["events"] - text_mix["safety_events"]
text_mix["text_label_structure"] = np.select(
    [text_mix["safety_events"].eq(0), text_mix["other_events"].eq(0)],
    ["비안전 사유만", "잠재 안전 사유만"],
    default="안전·비안전 혼합"
)
mix_summary = (
    text_mix.groupby("text_label_structure", as_index=False)
    .agg(texts=("normalized_text", "size"), events=("events", "sum"), safety_events=("safety_events", "sum"))
)
mix_summary["text_share_pct"] = mix_summary["texts"] / mix_summary["texts"].sum() * 100
mix_summary["event_share_pct"] = mix_summary["events"] / mix_summary["events"].sum() * 100
mix_summary["safety_event_share_pct"] = mix_summary["safety_events"] / mix_summary["safety_events"].sum() * 100
mix_summary.to_csv(TABLE_DIR / "16_same_text_mixed_feedback_structure.csv", index=False, encoding="utf-8-sig")
display(mix_summary)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
colors = {"비안전 사유만":"#8FAADC", "잠재 안전 사유만":"#E45756", "안전·비안전 혼합":"#F2C14E"}
for ax, value, title in zip(axes, ["event_share_pct", "safety_event_share_pct"], ["전체 피드백 사건", "잠재 안전 피드백"]):
    plot = mix_summary.set_index("text_label_structure")[value]
    ax.bar(plot.index, plot.values, color=[colors[i] for i in plot.index])
    ax.set_title(title)
    ax.set_ylabel("구성비(%)")
    ax.tick_params(axis="x", rotation=20)
plt.suptitle("동일 문구가 안전·비안전 사유를 함께 받는가")
savefig("16_same_text_mixed_feedback_structure.png")

test_frame = test_frame.copy()
test_frame["pred_prob"] = p_test
test_frame["pred"] = (p_test >= selected_threshold).astype(int)
test_frame["error_type"] = np.select(
    [(test_frame["potential_safety"].eq(1) & test_frame["pred"].eq(0)),
     (test_frame["potential_safety"].eq(0) & test_frame["pred"].eq(1))],
    ["false_negative", "false_positive"], default="correct"
)
error_examples = pd.concat([
    test_frame.loc[test_frame["error_type"].eq("false_negative")].sort_values("pred_prob").drop_duplicates("normalized_text").head(20),
    test_frame.loc[test_frame["error_type"].eq("false_positive")].sort_values("pred_prob", ascending=False).drop_duplicates("normalized_text").head(20),
])[["error_type", "question_text", "pred_prob", "potential_safety", "lex_any", "theme_label_ko"]]
error_examples.to_csv(TABLE_DIR / "16b_model_error_review_sample.csv", index=False, encoding="utf-8-sig")
display(error_examples.head(12))

동일 문구가 잠재 안전 사유와 다른 피드백 사유를 함께 받는지 확인한다. 혼합 비중이 높으면 텍스트 외 후보 관계와 노출 맥락이 추가로 필요하다.

## 14. 사전 기반 분류 한계

In [ ]:
lex_test = pair.loc[pair["split"].eq("test")].copy()
lex_precision = lex_test.loc[lex_test["lex_any"].eq(1), "potential_safety"].mean()
lex_recall = lex_test.loc[lex_test["potential_safety"].eq(1), "lex_any"].mean()
lex_flag_rate = lex_test["lex_any"].mean()
lex_lift = lex_precision / lex_test["potential_safety"].mean()
lex_screen = pd.DataFrame([{
    "test_rows": len(lex_test),
    "base_safety_share": lex_test["potential_safety"].mean(),
    "flag_rate": lex_flag_rate,
    "precision": lex_precision,
    "recall": lex_recall,
    "precision_lift_vs_base": lex_lift,
    "false_positive_share_among_flagged": 1 - lex_precision,
    "missed_safety_share": 1 - lex_recall,
}])
lex_screen.to_csv(TABLE_DIR / "17_lexicon_screening_limit.csv", index=False, encoding="utf-8-sig")
display(lex_screen)

missed = (
    lex_test.loc[lex_test["potential_safety"].eq(1) & lex_test["lex_any"].eq(0)]
    .groupby("normalized_text", as_index=False)
    .agg(question_text=("question_text", "first"), missed_safety_pairs=("potential_safety", "size"), theme_label_ko=("theme_label_ko", "first"))
    .sort_values("missed_safety_pairs", ascending=False).head(25)
)
false_flags = (
    lex_test.loc[lex_test["potential_safety"].eq(0) & lex_test["lex_any"].eq(1)]
    .groupby("normalized_text", as_index=False)
    .agg(question_text=("question_text", "first"), non_safety_pairs=("potential_safety", "size"), theme_label_ko=("theme_label_ko", "first"))
    .sort_values("non_safety_pairs", ascending=False).head(25)
)
missed.to_csv(TABLE_DIR / "17b_lexicon_missed_safety_examples.csv", index=False, encoding="utf-8-sig")
false_flags.to_csv(TABLE_DIR / "17c_lexicon_false_flag_examples.csv", index=False, encoding="utf-8-sig")
print("사전에 걸리지 않은 잠재 안전 표본")
display(missed.head(10))
print("사전에는 걸렸지만 비안전 사유였던 표본")
display(false_flags.head(10))

표현 사전의 정밀도, 재현율, 오탐, 미탐을 확인한다. 성능이 낮으면 자동 차단이 아닌 검수 우선순위에만 사용한다.

## 15. 민감도 분석

In [ ]:
# 같은 고정 사전에서 결과 방향이 분석 단위와 5월 편중에 따라 뒤집히는지 확인한다.
for category, words in LEXICON.items():
    linked[f"lex_{category}"] = linked["normalized_text"].map(lambda x: lexicon_flag(x, words))

sensitivity_rows = []
datasets = {
    "원행 전체": linked,
    "actor-text 중복제거": pair,
    "2023년 5월 원행": linked.loc[linked["event_month"].eq("2023-05")],
    "2023년 5월 actor-text": pair.loc[pair["event_month"].eq("2023-05")],
}
for data_name, frame in datasets.items():
    for category in LEXICON:
        col = f"lex_{category}"
        present = frame[col].eq(1)
        if present.sum() == 0:
            continue
        rate_with = frame.loc[present, "potential_safety"].mean()
        rate_without = frame.loc[~present, "potential_safety"].mean()
        sensitivity_rows.append({
            "dataset": data_name, "category": category,
            "rows_with_flag": int(present.sum()),
            "safety_share_with": rate_with,
            "safety_share_without": rate_without,
            "delta_pp": (rate_with - rate_without) * 100,
            "risk_ratio": rate_with / rate_without if rate_without else np.nan,
        })
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(TABLE_DIR / "18_lexicon_sensitivity_raw_dedup_may.csv", index=False, encoding="utf-8-sig")
pivot = sensitivity.pivot(index="category", columns="dataset", values="delta_pp")
display(pivot)

fig, ax = plt.subplots(figsize=(10.5, 6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label":"안전 구성비 차이(%p)"})
ax.set_title("분석 단위를 바꿔도 표현 범주의 방향이 유지되는가")
ax.set_xlabel(""); ax.set_ylabel("")
savefig("18_lexicon_sensitivity_heatmap.png")

## 16. 사람 검수 표본

In [ ]:
# 자동 규칙을 정답으로 확정하지 않도록 서로 다른 실패 유형을 섞은 검수 표본을 만든다.
review_parts = []
used_texts = set()

def add_review_sample(frame, stratum, n):
    candidates = frame.loc[~frame["normalized_text"].isin(used_texts)].copy()
    if candidates.empty:
        return
    take = candidates.sample(min(n, len(candidates)), random_state=SEED + len(review_parts))
    take["sample_stratum"] = stratum
    review_parts.append(take)
    used_texts.update(take["normalized_text"])

add_review_sample(text_stats.nlargest(200, "safety_pairs"), "잠재안전 건수 상위", 40)
add_review_sample(text_stats.loc[text_stats["feedback_pairs"].ge(10)].nlargest(200, "eb_safety_share"), "안전 구성비 상위", 40)
add_review_sample(text_stats.loc[text_stats["lex_any"].eq(1)], "핵심 표현 사전 플래그", 40)
add_review_sample(text_stats.loc[text_stats["lex_any"].eq(0) & text_stats["safety_pairs"].gt(0)], "사전 미탐 잠재안전", 40)
add_review_sample(text_stats, "무작위 대조", 40)

review_sample = pd.concat(review_parts, ignore_index=True)
review_sample = review_sample[[
    "sample_stratum", "question_text", "normalized_text", "feedback_pairs", "safety_pairs",
    "eb_safety_share", "theme_label_ko"
]]
review_sample["reviewer_a_category"] = ""
review_sample["reviewer_b_category"] = ""
review_sample["reviewer_a_harm_flag"] = ""
review_sample["reviewer_b_harm_flag"] = ""
review_sample["reviewer_note"] = ""
review_sample.to_csv(TABLE_DIR / "19_blind_manual_review_sample_200.csv", index=False, encoding="utf-8-sig")
print(f"블라인드 사람 검수용 표본 {len(review_sample)}개 생성")
display(review_sample.groupby("sample_stratum").size().rename("texts").to_frame())

상위 신호, 사전 미탐, 무작위 대조군을 포함한 검수 표본을 만든다. 두 검수자의 결과가 입력되면 Cohen's kappa와 범주별 정밀도를 계산할 수 있다.

## 17. 분석 QA

데이터 연결, 분할, 통계 검정, 해석 범위를 확인한다.

In [ ]:
qa_rows = [
    ("원천 질문 피드백 행 보존", "PASS" if len(feedback) == 51_424 else "FAIL", f"rows={len(feedback):,}"),
    ("질문 문구 연결", "PASS" if feedback["normalized_text"].notna().mean() > .999 else "WARN", f"link={feedback['normalized_text'].notna().mean():.4%}"),
    ("잠재 안전 라벨 분리", "PASS" if feedback["potential_safety"].sum() == 5_694 else "FAIL", f"potential_safety={feedback['potential_safety'].sum():,}"),
    ("actor-text 반복 제거 기록", "PASS", f"raw={len(linked):,}, pure_pairs={len(pair):,}, mixed={pair_raw['mixed_reason_pair'].sum():,}"),
    ("semantic family split 누수", "PASS" if pair.groupby('semantic_group_090')['split'].nunique().max() == 1 else "FAIL", "max split per family=" + str(pair.groupby('semantic_group_090')['split'].nunique().max())),
    ("Test 선행 사용 금지", "PASS", "어휘 후보=Train, 모델 선택=Validation, 최종 성능=Test"),
    ("노출 분모 사용 금지", "PASS", "모든 결과를 피드백 내 사유 구성으로 명명"),
    ("다중검정 보정", "PASS", "lexicon/token BH-FDR 적용"),
    ("실제 위해 확정 표현 금지", "PASS", "potential safety reason으로 제한"),
]
final_qa = pd.DataFrame(qa_rows, columns=["check", "status", "detail"])
final_qa.to_csv(TABLE_DIR / "99_final_qa.csv", index=False, encoding="utf-8-sig")
display(final_qa)

mix_lookup = mix_summary.set_index("text_label_structure")
mixed_safety_share = float(mix_lookup.loc["안전·비안전 혼합", "safety_event_share_pct"])
mixed_event_share = float(mix_lookup.loc["안전·비안전 혼합", "event_share_pct"])
evidence = pd.DataFrame([
    {
        "finding": "피드백 채널의 의미가 혼합됨",
        "evidence": f"개인 선호 {domain_counts.set_index('domain').loc['PERSONAL_PREFERENCE','share_pct']:.1f}%, 잠재 안전 {domain_counts.set_index('domain').loc['CONTENT_SAFETY','share_pct']:.1f}%",
        "strength": "관측 확인",
        "allowed_claim": "전체 질문 피드백을 안전 신고로 해석할 수 없다",
    },
    {
        "finding": "잠재 안전 신호가 일부 문구에 집중",
        "evidence": f"상위 1% 문구가 actor-text 잠재 안전 신호의 {safety_sorted.head(max(1,int(len(safety_sorted)*.01)))['safety_pairs'].sum()/safety_sorted['safety_pairs'].sum()*100:.1f}% 포함",
        "strength": "중복 사용자 제거 후 관측",
        "allowed_claim": "검수 우선순위 집중은 가능하나 노출 대비 위험률은 아님",
    },
    {
        "finding": "평가·추정 표현 범주와 잠재 안전 사유가 함께 나타남",
        "evidence": "감시, 체취·위생 희화, 친밀행동 추정, 외모·신체 평가 범주를 고정 Test와 조정모형으로 확인",
        "strength": "검증셋 연관",
        "allowed_claim": "자동 유해 판정이 아니라 검수 우선순위 신호",
    },
    {
        "finding": "같은 문구도 서로 다른 피드백을 받음",
        "evidence": f"혼합 문구가 전체 사건 {mixed_event_share:.1f}%, 잠재 안전 사건 {mixed_safety_share:.1f}%를 포함",
        "strength": "관측 확인",
        "allowed_claim": "문구 외 후보·관계·반복 맥락이 중요하다",
    },
    {
        "finding": "텍스트 단독 선별력은 제한적",
        "evidence": f"최종 {selected_name} Test PR-AUC {final_metrics['pr_auc']:.3f}, 상위10% 정밀도 {final_metrics['precision_at_top_10pct']:.3f}",
        "strength": "미사용 Test",
        "allowed_claim": "자동 차단이 아니라 검수 보조에 한정",
    },
])
evidence.to_csv(TABLE_DIR / "99b_final_evidence.csv", index=False, encoding="utf-8-sig")
display(evidence)

summary = {
    "rows_feedback": int(len(feedback)),
    "rows_potential_safety": int(feedback["potential_safety"].sum()),
    "linked_text_rate": float(feedback["normalized_text"].notna().mean()),
    "primary_actor_text_rows": int(len(pair)),
    "primary_safety_share": float(pair["potential_safety"].mean()),
    "selected_model": selected_name,
    "test_metrics": {k: float(v) for k, v in final_metrics.items()},
    "mixed_text_event_share_pct": mixed_event_share,
    "mixed_text_safety_event_share_pct": mixed_safety_share,
    "qa_fail_count": int(final_qa["status"].eq("FAIL").sum()),
}
(OUTPUT_DIR / "trust_text_analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))

### 해석 기준

- 신고 사유는 실제 위해 확정값이 아니라 사용자가 선택한 피드백 사유다.
- 질문 문구만으로 후보 관계와 노출 맥락을 알 수 없다.
- 텍스트 모델과 표현 사전은 검수 우선순위에만 사용한다.

## 18. 수동 카테고리 라벨

검수가 완료된 질문 카테고리 라벨을 불러온다. `vote`와 `미분류(질문 제거)` 행은 제외한다.

카테고리 라벨은 질문 주제 분류이며 잠재 안전 여부나 신고 사유가 아니다.


In [ ]:
CATEGORY_ORDER = [
    "외모_신체_평가",
    "외모_신체_비하",
    "성격_사회성",
    "관계_호감_친밀도",
    "연애",
    "능력_재능",
    "학교_학업",
    "음주_폭력",
    "취향_일상",
    "매력",
]

manual_labels = pd.read_csv(
    MANUAL_LABEL_PATH,
    dtype={"question_id": "string", "question": "string", "label": "string"},
    keep_default_na=False,
    encoding="utf-8-sig",
)

required_columns = ["question_id", "question", "label"]
if manual_labels.columns.tolist() != required_columns:
    raise ValueError(f"수동 라벨 컬럼이 다릅니다: {manual_labels.columns.tolist()}")

manual_labels["label"] = manual_labels["label"].str.strip()
manual_labels["normalized_text"] = manual_labels["question"].map(normalize_text)

invalid_labels = sorted(set(manual_labels["label"]) - set(CATEGORY_ORDER))
manual_label_qa = pd.DataFrame(
    [
        ("전체 질문", len(manual_labels)),
        ("정규화 고유 문구", manual_labels["normalized_text"].nunique()),
        ("중복 정규화 문구", manual_labels.duplicated("normalized_text").sum()),
        ("빈 라벨", manual_labels["label"].eq("").sum()),
        ("허용되지 않은 라벨", len(invalid_labels)),
    ],
    columns=["check", "value"],
)

assert not manual_labels.empty
assert manual_labels["normalized_text"].is_unique
assert not manual_labels["label"].eq("").any()
assert not invalid_labels

manual_label_qa.to_csv(
    TABLE_DIR / "manual_label_qa.csv", index=False, encoding="utf-8-sig"
)
display(manual_label_qa)
display(manual_labels["label"].value_counts().reindex(CATEGORY_ORDER, fill_value=0).rename("questions").to_frame())

## 19. 피드백 사건과 수동 라벨 연결

여러 질문 ID가 한 행에 묶인 경우가 있어 `normalized_text`를 연결 키로 사용한다.


In [ ]:
manual_label_map = manual_labels[
    ["normalized_text", "question_id", "question", "label"]
].rename(
    columns={
        "question_id": "manual_question_ids",
        "question": "manual_question_text",
        "label": "manual_category",
    }
)

feedback_labeled = feedback.merge(
    manual_label_map,
    on="normalized_text",
    how="left",
    validate="many_to_one",
)

linkage_qa = pd.DataFrame(
    [
        ("질문 피드백 사건", len(feedback_labeled)),
        ("수동 라벨 연결 사건", feedback_labeled["manual_category"].notna().sum()),
        ("수동 라벨 미연결 사건", feedback_labeled["manual_category"].isna().sum()),
        ("연결된 고유 문구", feedback_labeled.loc[feedback_labeled["manual_category"].notna(), "normalized_text"].nunique()),
    ],
    columns=["check", "value"],
)
linkage_qa["share"] = linkage_qa["value"] / len(feedback_labeled)

question_text_labeled = (
    feedback_labeled.dropna(subset=["manual_category"])
    .groupby(["normalized_text", "manual_category"], as_index=False)
    .agg(
        question_text=("question_text", "first"),
        feedback_events=("potential_safety", "size"),
        potential_safety_events=("potential_safety", "sum"),
        actors=("actor_user_id", "nunique"),
    )
)
question_text_labeled["potential_safety_share"] = (
    question_text_labeled["potential_safety_events"]
    / question_text_labeled["feedback_events"]
)

linkage_qa.to_csv(TABLE_DIR / "manual_label_linkage_qa.csv", index=False, encoding="utf-8-sig")
question_text_labeled.to_csv(
    TABLE_DIR / "manual_labeled_question_text_analysis_base.csv",
    index=False,
    encoding="utf-8-sig",
)

display(linkage_qa)
display(question_text_labeled.head())

## 20. 카테고리별 요약

질문 수와 피드백 사건 수를 분리한다. `potential_safety_share`는 피드백 내부의 잠재 안전 사유 구성비다.


In [ ]:
category_summary = (
    feedback_labeled.dropna(subset=["manual_category"])
    .groupby("manual_category", as_index=False)
    .agg(
        questions=("normalized_text", "nunique"),
        feedback_events=("potential_safety", "size"),
        actors=("actor_user_id", "nunique"),
        potential_safety_events=("potential_safety", "sum"),
    )
)
category_summary["question_share"] = category_summary["questions"] / category_summary["questions"].sum()
category_summary["feedback_event_share"] = (
    category_summary["feedback_events"] / category_summary["feedback_events"].sum()
)
category_summary["potential_safety_share"] = (
    category_summary["potential_safety_events"] / category_summary["feedback_events"]
)
category_summary["manual_category"] = pd.Categorical(
    category_summary["manual_category"], categories=CATEGORY_ORDER, ordered=True
)
category_summary = category_summary.sort_values("manual_category").reset_index(drop=True)

category_summary.to_csv(
    TABLE_DIR / "manual_category_feedback_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
display(category_summary)

## 21. 후속 분석용 데이터

- `feedback_labeled`: 피드백 사건 단위 데이터
- `question_text_labeled`: 질문 문구 단위 데이터
- 후속 모델도 semantic family 기준 Train / Validation / Test 분할을 유지한다.
- Test 결과를 기준으로 특징이나 임계값을 다시 선택하지 않는다.
